In [1]:
# === 05: PREDIKSI & ERROR (FINAL) ===
from pathlib import Path
import json
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.metrics import mean_absolute_error, mean_squared_error

# --- Konfigurasi ---
ROOT = Path.cwd().parent
DATA_CLEAN = ROOT / "data_clean"
MODELS = ROOT / "models"
OUTPUTS = ROOT / "outputs"; OUTPUTS.mkdir(exist_ok=True)

CLEAN_FILE = DATA_CLEAN / "dataset_final_2021-2024.csv"
MODEL_FILE = MODELS / "ols_model_final.pkl"
FEAT_FILE  = MODELS / "selected_features.json"
TARGET   = "P1"
TIME_COL = "Tahun"
TEST_YEAR = 2024

# --- Validasi artefak ---
assert MODEL_FILE.exists(), f"Model tidak ditemukan: {MODEL_FILE}"
assert FEAT_FILE.exists(),  f"Daftar fitur tidak ditemukan: {FEAT_FILE}"
assert CLEAN_FILE.exists(), f"Data bersih tidak ditemukan: {CLEAN_FILE}"

# --- Load data & artefak ---
df = pd.read_csv(CLEAN_FILE)
res = sm.load(str(MODEL_FILE))
with open(FEAT_FILE) as f:
    selected_cols = json.load(f)

# --- Siapkan test set ---
df_test = df[df[TIME_COL] == TEST_YEAR].copy()
need = [c for c in selected_cols if c != "const"]

missing = [c for c in need if c not in df_test.columns]
if missing:
    raise ValueError(f"Kolom hilang di data test: {missing}")

# pastikan semua fitur numerik
df_test[need] = df_test[need].apply(pd.to_numeric, errors="coerce")
if df_test[need].isna().any().any():
    raise ValueError("Ada NaN di fitur test setelah konversi numerik. Bersihkan dulu data test.")

X_test = sm.add_constant(df_test[need], has_constant="add")
# susun tepat sesuai urutan training
X_test = X_test[selected_cols] if "const" in selected_cols else X_test[["const"] + need]

# --- Prediksi & metrik ---
y_true = df_test[TARGET].values
y_pred = res.predict(X_test).values

mae  = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mape = np.nanmean(np.where(y_true != 0, np.abs((y_true - y_pred) / y_true) * 100, np.nan))

print(f"MAE  = {mae:.4f}")
print(f"RMSE = {rmse:.4f}")
print(f"MAPE = {mape:.2f}%")

# --- Simpan output ---
out_csv = OUTPUTS / f"prediksi_{TEST_YEAR}.csv"
df_out = df_test.copy()
df_out["prediksi"]  = y_pred
df_out["abs_error"] = np.abs(df_out[TARGET] - df_out["prediksi"])
df_out["ape_%"]     = np.where(df_out[TARGET] != 0, df_out["abs_error"] / df_out[TARGET] * 100, np.nan)
df_out.to_csv(out_csv, index=False)

meta = {
    "model_file": str(MODELS / "ols_model_final.pkl"),
    "features_file": str(MODELS / "selected_features.json"),
    "target": TARGET,
    "test_year": TEST_YEAR,
    "metrics": {"MAE": float(mae), "RMSE": float(rmse), "MAPE_%": float(mape)}
}
with open(OUTPUTS / f"ringkasan_error_{TEST_YEAR}.json", "w") as f:
    json.dump(meta, f, indent=2)

print("Saved:", out_csv)
print("Saved:", OUTPUTS / f"ringkasan_error_{TEST_YEAR}.json")


MAE  = 0.4205
RMSE = 0.5121
MAPE = 33.72%
Saved: c:\Users\ASUS\Documents\KULIAH\VS Code\projek-PDS-IPM\outputs\prediksi_2024.csv
Saved: c:\Users\ASUS\Documents\KULIAH\VS Code\projek-PDS-IPM\outputs\ringkasan_error_2024.json
